# Astrometric offset: before vs after concordance

Measures Rubin-vs-VIS and NISP-vs-VIS astrometric shifts across all tiles **before** and
**after** applying the trained concordance correction.

Left panel: raw WCS offsets (should reproduce the ~50-100 mas contours).  
Right panel: residuals after concordance correction (should shrink dramatically).

All 9 target bands (6 Rubin + 3 NISP) are shown when available.

In [ ]:
import sys, glob, os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from astropy.io.fits import Header
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord, search_around_sky
from astropy import units as u
from scipy.ndimage import maximum_filter, median_filter, zoom, gaussian_filter

# Project root
ROOT = Path('..').resolve()
RUBIN_DIR  = ROOT / 'data' / 'rubin_tiles_all'
EUCLID_DIR = ROOT / 'data' / 'euclid_tiles_all'

# Set this to the concordance FITS once available:
CONCORDANCE_FITS = None   # e.g. ROOT / 'checkpoints' / 'astro_v7_centernet_200tiles' / 'concordance_all.fits'

RUBIN_BANDS = ['u', 'g', 'r', 'i', 'z', 'y']
NISP_BANDS  = ['Y', 'J', 'H']
ALL_TARGET_BANDS = RUBIN_BANDS + NISP_BANDS

rtiles = sorted(RUBIN_DIR.glob('tile_x*_y*.npz'))
rtiles = [p for p in rtiles if not p.name.endswith('_euclid.npz')]
print(f'Rubin tiles:  {len(rtiles)}')
print(f'Euclid tiles: {len(list(EUCLID_DIR.glob("tile_x*_y*_euclid.npz")))}')

## 1. Source detection & cross-matching helpers

In [2]:
RUBIN_BANDS = ['u', 'g', 'r', 'i', 'z', 'y']

def find_peaks(img, nsigma=5.0, border=10, top=2000, local_box=64):
    """Local-MAD robust peak detection."""
    data = np.nan_to_num(img, nan=-np.inf)
    ny, nx = data.shape
    b = int(local_box)
    py, px = (-ny) % b, (-nx) % b
    data_p = np.pad(data, ((0, py), (0, px)), mode='edge') if (py or px) else data
    Ny, Nx = data_p.shape
    by, bx = Ny // b, Nx // b
    blocks = data_p.reshape(by, b, bx, b)
    m_blk = np.nanmedian(blocks, axis=(1, 3))
    mad_blk = np.nanmedian(np.abs(blocks - m_blk[:, None, :, None]), axis=(1, 3))
    s_blk = 1.4826 * mad_blk
    m = zoom(m_blk, (b, b), order=1)[:Ny, :Nx][:ny, :nx]
    s = zoom(s_blk, (b, b), order=1)[:Ny, :Nx][:ny, :nx]
    thr = m + nsigma * s
    local_max = maximum_filter(data, size=3) == data
    mask = (data > thr) & local_max & np.isfinite(data)
    if border:
        mask[:border, :] = False; mask[-border:, :] = False
        mask[:, :border] = False; mask[:, -border:] = False
    ys, xs = np.where(mask)
    if xs.size == 0:
        return np.array([]), np.array([])
    vals = data[ys, xs]
    order = np.argsort(vals)[::-1]
    if top is not None:
        order = order[:top]
    return xs[order].astype(float), ys[order].astype(float)


def _parabolic_subpixel_1d(fm1, f0, fp1):
    denom = (fm1 - 2.0 * f0 + fp1)
    if denom == 0 or not np.isfinite(denom):
        return 0.0
    return float(np.clip(0.5 * (fm1 - fp1) / denom, -0.75, 0.75))


def refine_centroids(img, xs, ys, r=5):
    outx, outy = [], []
    ny, nx = img.shape
    data = np.nan_to_num(img, nan=-np.inf)
    for x0, y0 in zip(xs, ys):
        ix, iy = int(round(x0)), int(round(y0))
        y1, y2 = max(0, iy - r), min(ny, iy + r + 1)
        x1, x2 = max(0, ix - r), min(nx, ix + r + 1)
        cut = data[y1:y2, x1:x2]
        if cut.size == 0 or not np.isfinite(cut).any():
            outx.append(x0); outy.append(y0); continue
        bkg = np.nanmedian(cut[np.isfinite(cut)])
        w = np.clip(cut - bkg, 0, None)
        if np.sum(w) <= 0:
            outx.append(x0); outy.append(y0); continue
        py_, px_ = np.unravel_index(np.argmax(w), w.shape)
        if py_ <= 0 or py_ >= w.shape[0]-1 or px_ <= 0 or px_ >= w.shape[1]-1:
            outx.append(x1 + px_); outy.append(y1 + py_); continue
        f0 = w[py_, px_]
        dx = _parabolic_subpixel_1d(w[py_, px_-1], f0, w[py_, px_+1])
        dy = _parabolic_subpixel_1d(w[py_-1, px_], f0, w[py_+1, px_])
        outx.append(x1 + px_ + dx)
        outy.append(y1 + py_ + dy)
    return np.array(outx), np.array(outy)


def nn_greedy_unique(ref_sky, cand_sky, max_sep_arcsec=0.15):
    """One-to-one greedy nearest-neighbour match. Returns (ri, ci, sep_arcsec)."""
    if len(ref_sky) == 0 or len(cand_sky) == 0:
        return np.array([],dtype=int), np.array([],dtype=int), np.array([])
    i_ref, i_cand, sep2d, _ = search_around_sky(ref_sky, cand_sky, max_sep_arcsec * u.arcsec)
    if len(i_ref) == 0:
        return np.array([],dtype=int), np.array([],dtype=int), np.array([])
    sep = sep2d.to(u.arcsec).value
    order = np.argsort(sep)
    used_r, used_c = set(), set()
    ri, ci, si = [], [], []
    for k in order:
        r, c = int(i_ref[k]), int(i_cand[k])
        if r in used_r or c in used_c:
            continue
        used_r.add(r); used_c.add(c)
        ri.append(r); ci.append(c); si.append(sep[k])
    return np.array(ri), np.array(ci), np.array(si)

## 2. Load all tiles, detect sources, cross-match vs VIS

For Rubin bands: detect in each band independently, match against VIS detections.  
For NISP bands: detect in each NISP band, match against VIS detections using their own WCS.

In [ ]:
def load_wcs_from_header(hdr_data):
    """Parse WCS from the stored header (dict or string)."""
    if isinstance(hdr_data, np.ndarray):
        hdr_data = hdr_data.item()
    if isinstance(hdr_data, str):
        return WCS(Header.fromstring(hdr_data))
    if isinstance(hdr_data, dict):
        return WCS(hdr_data)
    return WCS(Header(hdr_data))


def pix_to_sky(wcs, xs, ys):
    """Pixel (x, y) arrays -> SkyCoord."""
    radec = wcs.all_pix2world(np.column_stack([xs, ys]), 0)
    return SkyCoord(ra=radec[:, 0]*u.deg, dec=radec[:, 1]*u.deg, frame='icrs')


def measure_offsets_for_band(band_img, band_wcs, ref_sky, vis_wcs,
                              nsigma=10, top=3000, local_box=64,
                              max_sep=0.15):
    """Detect sources in a band image, match to VIS reference, return (dRA, dDec) in mas."""
    bxs, bys = find_peaks(band_img, nsigma=nsigma, top=top, local_box=local_box)
    if bxs.size < 3:
        return None
    bxs, bys = refine_centroids(band_img, bxs, bys, r=5)
    cand_sky = pix_to_sky(band_wcs, bxs, bys)

    ri, ci, sep = nn_greedy_unique(ref_sky, cand_sky, max_sep_arcsec=max_sep)
    if len(ri) < 3:
        return None

    ref_m  = ref_sky[ri]
    cand_m = cand_sky[ci]
    dra  = (cand_m.ra.deg  - ref_m.ra.deg) * np.cos(np.deg2rad(ref_m.dec.deg)) * 3600e3
    ddec = (cand_m.dec.deg - ref_m.dec.deg) * 3600e3
    return np.column_stack([dra, ddec])


# Accumulate offsets: band -> list of (dra_mas, ddec_mas) arrays
offsets_raw = {b: [] for b in ALL_TARGET_BANDS}
n_tiles = len(rtiles)

for ti, rpath in enumerate(rtiles):
    tile_id = rpath.stem
    epath = EUCLID_DIR / f'{tile_id}_euclid.npz'
    if not epath.exists():
        continue

    rdata = np.load(str(rpath), allow_pickle=True)
    edata = np.load(str(epath), allow_pickle=True)

    # VIS reference catalog
    vis_img = np.nan_to_num(edata['img_VIS'].astype(np.float32), nan=0.0)
    vis_wcs = load_wcs_from_header(edata['wcs_VIS'])
    vxs, vys = find_peaks(vis_img, nsigma=10, top=3000, local_box=128)
    if vxs.size < 5:
        continue
    vxs, vys = refine_centroids(vis_img, vxs, vys, r=5)
    ref_sky = pix_to_sky(vis_wcs, vxs, vys)

    # --- Rubin bands ---
    rubin_wcs = load_wcs_from_header(rdata['wcs_hdr'])
    rubin_cube = rdata['img']  # [6, 512, 512]

    for bi, bname in enumerate(RUBIN_BANDS):
        if bi >= rubin_cube.shape[0]:
            continue
        band_img = np.nan_to_num(rubin_cube[bi].astype(np.float32), nan=0.0)
        result = measure_offsets_for_band(band_img, rubin_wcs, ref_sky, vis_wcs)
        if result is not None:
            offsets_raw[bname].append(result)

    # --- NISP bands ---
    for nb in NISP_BANDS:
        img_key = f'img_{nb}'
        wcs_key = f'wcs_{nb}'
        if img_key not in edata or wcs_key not in edata:
            continue
        try:
            nisp_img = np.nan_to_num(edata[img_key].astype(np.float32), nan=0.0)
            nisp_wcs = load_wcs_from_header(edata[wcs_key])
        except Exception:
            continue
        # NISP has coarser pixels (0.3"/px), use looser detection and matching
        result = measure_offsets_for_band(
            nisp_img, nisp_wcs, ref_sky, vis_wcs,
            nsigma=8, top=2000, local_box=32, max_sep=0.3)
        if result is not None:
            offsets_raw[nb].append(result)

    if (ti + 1) % 50 == 0 or ti == 0:
        print(f'  [{ti+1}/{n_tiles}] {tile_id}')

# Concatenate
for b in ALL_TARGET_BANDS:
    if offsets_raw[b]:
        offsets_raw[b] = np.concatenate(offsets_raw[b], axis=0)
    else:
        offsets_raw[b] = np.zeros((0, 2))

print('\nMatched sources per band (raw WCS):')
for b in ALL_TARGET_BANDS:
    n = len(offsets_raw[b])
    if n > 0:
        rms = np.sqrt(np.mean(offsets_raw[b]**2))
        print(f'  {b:>5s}: {n:5d} matches, RMS={rms:.1f} mas')
    else:
        print(f'  {b:>5s}: 0 matches')

## 3. (Optional) Apply concordance and re-measure

In [ ]:
offsets_corrected = None

if CONCORDANCE_FITS is not None and Path(CONCORDANCE_FITS).exists():
    sys.path.insert(0, str(ROOT / 'models' / 'astrometry2'))
    from apply_concordance import ConcordanceMap
    cmap = ConcordanceMap(str(CONCORDANCE_FITS))

    offsets_corrected = {b: [] for b in ALL_TARGET_BANDS}

    for ti, rpath in enumerate(rtiles):
        tile_id = rpath.stem
        epath = EUCLID_DIR / f'{tile_id}_euclid.npz'
        if not epath.exists():
            continue

        rdata = np.load(str(rpath), allow_pickle=True)
        edata = np.load(str(epath), allow_pickle=True)

        vis_img = np.nan_to_num(edata['img_VIS'].astype(np.float32), nan=0.0)
        vis_wcs = load_wcs_from_header(edata['wcs_VIS'])
        vxs, vys = find_peaks(vis_img, nsigma=10, top=3000, local_box=128)
        if vxs.size < 5:
            continue
        vxs, vys = refine_centroids(vis_img, vxs, vys, r=5)
        ref_sky = pix_to_sky(vis_wcs, vxs, vys)

        rubin_wcs = load_wcs_from_header(rdata['wcs_hdr'])
        rubin_cube = rdata['img']

        # --- Rubin bands ---
        for bi, bname in enumerate(RUBIN_BANDS):
            if bi >= rubin_cube.shape[0]:
                continue
            band_img = np.nan_to_num(rubin_cube[bi].astype(np.float32), nan=0.0)
            bxs, bys = find_peaks(band_img, nsigma=10, top=3000, local_box=64)
            if bxs.size < 3:
                continue
            bxs, bys = refine_centroids(band_img, bxs, bys, r=5)

            try:
                corrected_vis_x, corrected_vis_y = cmap.rubin_to_vis(
                    bxs, bys, rubin_wcs, vis_wcs, tile_id, bname)
                cand_sky = pix_to_sky(vis_wcs, corrected_vis_x, corrected_vis_y)
            except Exception:
                continue

            ri, ci, sep = nn_greedy_unique(ref_sky, cand_sky, max_sep_arcsec=0.15)
            if len(ri) < 3:
                continue

            ref_m  = ref_sky[ri]
            cand_m = cand_sky[ci]
            dra  = (cand_m.ra.deg  - ref_m.ra.deg) * np.cos(np.deg2rad(ref_m.dec.deg)) * 3600e3
            ddec = (cand_m.dec.deg - ref_m.dec.deg) * 3600e3
            offsets_corrected[bname].append(np.column_stack([dra, ddec]))

        # --- NISP bands ---
        for nb in NISP_BANDS:
            img_key = f'img_{nb}'
            wcs_key = f'wcs_{nb}'
            if img_key not in edata or wcs_key not in edata:
                continue
            try:
                nisp_img = np.nan_to_num(edata[img_key].astype(np.float32), nan=0.0)
                nisp_wcs = load_wcs_from_header(edata[wcs_key])
            except Exception:
                continue

            bxs, bys = find_peaks(nisp_img, nsigma=8, top=2000, local_box=32)
            if bxs.size < 3:
                continue
            bxs, bys = refine_centroids(nisp_img, bxs, bys, r=5)

            # Apply concordance: NISP pixel -> sky -> corrected sky -> VIS pixel
            try:
                # Convert NISP pixels to sky
                nisp_ra, nisp_dec = nisp_wcs.wcs_pix2world(bxs, bys, 0)
                # Apply concordance correction at those sky positions
                dra_corr, ddec_corr = cmap.correction_at_sky(nisp_ra, nisp_dec, tile_id, nb)
                cos_dec = np.cos(np.deg2rad(nisp_dec))
                corr_ra  = nisp_ra  + (dra_corr / 3600.0) / cos_dec
                corr_dec = nisp_dec + (ddec_corr / 3600.0)
                # Project corrected sky to VIS pixel frame
                corrected_vis_x, corrected_vis_y = vis_wcs.wcs_world2pix(corr_ra, corr_dec, 0)
                cand_sky = pix_to_sky(vis_wcs, corrected_vis_x, corrected_vis_y)
            except Exception:
                continue

            ri, ci, sep = nn_greedy_unique(ref_sky, cand_sky, max_sep_arcsec=0.3)
            if len(ri) < 3:
                continue

            ref_m  = ref_sky[ri]
            cand_m = cand_sky[ci]
            dra  = (cand_m.ra.deg  - ref_m.ra.deg) * np.cos(np.deg2rad(ref_m.dec.deg)) * 3600e3
            ddec = (cand_m.dec.deg - ref_m.dec.deg) * 3600e3
            offsets_corrected[nb].append(np.column_stack([dra, ddec]))

        if (ti + 1) % 50 == 0 or ti == 0:
            print(f'  [{ti+1}/{n_tiles}] {tile_id}')

    for b in ALL_TARGET_BANDS:
        if offsets_corrected[b]:
            offsets_corrected[b] = np.concatenate(offsets_corrected[b], axis=0)
        else:
            offsets_corrected[b] = np.zeros((0, 2))

    print('\nMatched sources per band (after concordance):')
    for b in ALL_TARGET_BANDS:
        n = len(offsets_corrected[b])
        if n > 0:
            rms = np.sqrt(np.mean(offsets_corrected[b]**2))
            print(f'  {b:>5s}: {n:5d} matches, RMS={rms:.1f} mas')
else:
    print('No concordance FITS set. Run infer_concordance.py first, then set CONCORDANCE_FITS above.')

## 4. Plot: before vs after contours

In [ ]:
BAND_COLORS = {
    'u': '#0B1F3A', 'g': '#143A6F', 'r': '#1F5FA3',
    'i': '#4B8FD1', 'z': '#8FC6FF', 'y': '#00FFFF',
    'Y': '#FF6600', 'J': '#FF3300', 'H': '#CC0000',
}


def _mad_sigma(x):
    return 1.4826 * np.median(np.abs(x - np.median(x)))


def plot_contours(ax, offsets_dict, title, view_limit=150,
                  clip_sigma=3.5, n_bins=45, smooth_sigma=2.2):
    """Draw astrometric offset contours on ax."""
    ax.set_xlim(-view_limit, view_limit)
    ax.set_ylim(-view_limit, view_limit)
    ax.axhline(0, color='gray', ls='--', lw=0.5)
    ax.axvline(0, color='gray', ls='--', lw=0.5)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    ax.set_xlabel('dRA [mas]')
    ax.set_ylabel('dDec [mas]')
    ax.set_title(title)

    handles = []
    for bname in ALL_TARGET_BANDS:
        pts = offsets_dict[bname]
        if len(pts) < 30:
            continue
        dra, ddec = pts[:, 0], pts[:, 1]

        # Robust clip
        med_ra, sig_ra = np.median(dra), _mad_sigma(dra)
        med_de, sig_de = np.median(ddec), _mad_sigma(ddec)
        keep = (
            (np.abs(dra - med_ra) < clip_sigma * sig_ra) &
            (np.abs(ddec - med_de) < clip_sigma * sig_de)
        )
        dra_k, ddec_k = dra[keep], ddec[keep]
        if len(dra_k) < 30:
            continue

        color = BAND_COLORS.get(bname, 'gray')
        # Solid lines for Rubin, dashed for NISP
        ls = '-' if bname in RUBIN_BANDS else '--'

        # 2D histogram -> smoothed contour
        H, xedges, yedges = np.histogram2d(
            dra_k, ddec_k, bins=n_bins,
            range=[[-view_limit, view_limit], [-view_limit, view_limit]])
        H = gaussian_filter(H.T, sigma=smooth_sigma)
        H = H / H.max() if H.max() > 0 else H
        X = 0.5 * (xedges[:-1] + xedges[1:])
        Y = 0.5 * (yedges[:-1] + yedges[1:])
        level = np.exp(-0.5)  # 1-sigma contour
        ax.contour(X, Y, H, levels=[level], colors=color, linewidths=2.5, linestyles=ls)

        import matplotlib.lines as mlines
        prefix = 'NISP ' if bname in NISP_BANDS else ''
        handles.append(mlines.Line2D([], [], color=color, lw=2.5, ls=ls,
                                      label=f'{prefix}{bname} (n={len(dra_k)})'))

    ax.legend(handles=handles, fontsize=7, loc='upper left')


# --- Plot ---
n_panels = 2 if offsets_corrected is not None else 1
fig, axes = plt.subplots(1, n_panels, figsize=(7 * n_panels, 6.5))
if n_panels == 1:
    axes = [axes]

plot_contours(axes[0], offsets_raw, 'Before concordance (raw WCS)')

if offsets_corrected is not None:
    plot_contours(axes[1], offsets_corrected, 'After concordance')

plt.tight_layout()
plt.savefig('astrometry_before_after.png', dpi=200, bbox_inches='tight')
plt.show()

## 5. Summary statistics

In [ ]:
print(f"{'Band':<8} {'N':>6} {'med_off':>10} {'RMS':>10} {'MAD_RA':>10} {'MAD_Dec':>10}")
print(f"{'':8s} {'':6s} {'[mas]':>10} {'[mas]':>10} {'[mas]':>10} {'[mas]':>10}")
print('-' * 62)

for label, odict in [('RAW', offsets_raw), ('CONC', offsets_corrected)]:
    if odict is None:
        continue
    print(f'\n--- {label} ---')
    for b in ALL_TARGET_BANDS:
        pts = odict[b]
        prefix = 'NISP_' if b in NISP_BANDS else ''
        bname = f'{prefix}{b}'
        if len(pts) < 3:
            print(f'  {bname:<8} {len(pts):>6}')
            continue
        dra, ddec = pts[:, 0], pts[:, 1]
        med_off = np.hypot(np.median(dra), np.median(ddec))
        rms = np.sqrt(np.mean(dra**2 + ddec**2))
        print(f'  {bname:<8} {len(pts):>6} {med_off:>10.1f} {rms:>10.1f} '
              f'{_mad_sigma(dra):>10.1f} {_mad_sigma(ddec):>10.1f}')